### Graficas o insights para Financiación ICETEX (créditos académicos, financieros y de continuidad)

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sbn
import numpy as np

In [ ]:
# cargamos el dataset
ROOT = Path('..').resolve()
icetex = pd.read_csv(ROOT/'icetex.csv', sep=';')
display(icetex.head())

In [ ]:
# Exploramos las columnas y datos que tenemos así como su tipo
display(icetex.info())
display(icetex['ACADE'].value_counts())
display(icetex['FINAN'].value_counts())
display(icetex['CI_RECI'].value_counts())
display(icetex['TIPO'].value_counts())

A diferencia de los datasets anteriores, aquí no hay una columna identitaria como `SEXO` que filtrar — `ACADE`, `FINAN` y `CI_RECI` son las variables de interés en sí mismas (rangos de créditos ICETEX recibidos: académicos, financieros/subsistencia, y de continuidad). Cada combinación de las 3 está presente (incluyendo "Sin Información"), así que no eliminamos filas, vamos directo al melt.

Hacemos la conversión de formato ancho a formato largo

In [ ]:
columnas_stay = ['ACADE','FINAN','CI_RECI','TIPO']
columnas_melt = [col for col in icetex.columns if col not in columnas_stay]
print(columnas_melt)
long_icetex = icetex.melt(
    id_vars=columnas_stay,
    value_vars=columnas_melt,       # lista de '2015-1', '2015-2', ...
    var_name='periodo',
    value_name='valor'
)
display(long_icetex.head())

Verificamos NaNs o nulls para limpiar

In [ ]:
print(long_icetex.isnull().sum())

In [ ]:
pivot = long_icetex.pivot_table(
    index=['ACADE','FINAN','CI_RECI','periodo'],
    columns='TIPO',
    values='valor',
    aggfunc=lambda x: x.sum(min_count=1)
).reset_index()
display(pivot.head())

Calculamos la tasa de deserción y corregimos outliers (mismo criterio que en los notebooks anteriores: acotamos a [0, 100])

In [ ]:
pivot['tasa'] = round((pivot['DESERTORES'] / pivot['MATRICULADOS']) * 100, 2)
pivot['tasa'] = np.clip(pivot['tasa'], 0, 100)
display(pivot.head())

### Gráficas

Las categorías de `ACADE`, `FINAN` y `CI_RECI` tienen un orden lógico (`Ninguno` < `1 a 3` < `4 a 6` < `Más de 7`), así que las ordenamos explícitamente en el eje X en vez de dejar el orden alfabético por defecto.

In [ ]:
orden_creditos = ['Ninguno', '1 a 3', '4 a 6', 'M¿s de 7', 'Sin Informaci¿n']

fig, axs = plt.subplots(2, 2, figsize=(15, 12))

sbn.barplot(data=pivot, x='ACADE', y='tasa', order=orden_creditos, ax=axs[0,0], color='steelblue')
axs[0,0].tick_params(axis='x', rotation=45)
axs[0,0].set_title('Tasa de deserción por créditos académicos (ACADE)')

sbn.barplot(data=pivot, x='FINAN', y='tasa', order=orden_creditos, ax=axs[0,1], color='darkorange')
axs[0,1].tick_params(axis='x', rotation=45)
axs[0,1].set_title('Tasa de deserción por créditos financieros (FINAN)')

sbn.barplot(data=pivot, x='CI_RECI', y='tasa', order=orden_creditos, ax=axs[1,0], color='seagreen')
axs[1,0].tick_params(axis='x', rotation=45)
axs[1,0].set_title('Tasa de deserción por créditos de continuidad (CI_RECI)')

pivot['recibio_credito'] = np.where(
    (pivot['ACADE'] == 'Ninguno') & (pivot['FINAN'] == 'Ninguno'),
    'Sin crédito ICETEX',
    'Con algún crédito ICETEX'
)
sbn.barplot(data=pivot, x='recibio_credito', y='tasa', ax=axs[1,1], color='indianred')
axs[1,1].set_title('Tasa de deserción: con vs. sin crédito ICETEX')

plt.tight_layout()
plt.show()

Evolución de la tasa de deserción por créditos académicos (ACADE) a través del tiempo

In [ ]:
f = pivot.groupby(['ACADE','periodo'])['tasa'].mean().reset_index()

# Forzamos orden cronológico explícito en el eje X (lección aprendida en
# sex_programa.ipynb: lineplot ordena por orden de aparición, no alfabético,
# cuando las categorías no arrancan todas en el mismo período)
orden_periodos = sorted(f['periodo'].unique())
f['periodo'] = pd.Categorical(f['periodo'], categories=orden_periodos, ordered=True)
f = f.sort_values('periodo')

plt.figure(figsize=(18,6))
ax = sbn.lineplot(data=f, x='periodo', y='tasa', hue='ACADE', hue_order=orden_creditos, marker='o')

for i, label in enumerate(ax.get_xticklabels()):
    if i % 2 != 0:
        label.set_visible(False)

plt.xticks(rotation=90)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()